# Reference-based spatial deconvolution

Estimate contributions from named cell types using your target counts and fixed reference profiles.
Start with `counts.h5ad` and an annotated cells × genes `scrna.h5ad`. The preparation cells
below build `reference.h5ad`: cell types × marker genes, with unnormalized per-cell reference
signals. You can also load an existing reference. Target coordinates go in `obsm["spatial"]`.

The reference remains fixed during fitting. Dividing molecular weights by type-specific reference
signal yields produces estimated cell fractions, assuming those relative yields represent the target.
This example uses the CPU fixed-reference solver with spatial weight zero.

## Data links

- Target: [10x Xenium human lymph node](https://www.10xgenomics.com/datasets/preview-data-xenium-prime-gene-expression).
- Single-cell reference counts: [10x Flex, 320k scFFPE from eight human tissues](https://www.10xgenomics.com/datasets/320k_scFFPE_16-plex_GEM-X_FLEX), sample `LNReactive_BC9-10`.

Supply a cell-type annotation column for the single-cell reference. The study used locally
annotated cell groups; the count download alone does not supply those study-specific labels.

## Example figures

Archived Xenium tutorial figures: estimated B-, CD4 T- and CD8 T-cell fractions and
agreement with annotated groups. These restored figures used the earlier COSG top100 panel;
the reference-preparation example below implements the study's updated top50 rule.

![Estimated B, CD4 T and CD8 T fractions in Xenium pseudo-spots](../docs/source/_static/figures/xenium_maps.png)

![Xenium estimated and annotated group-fraction correlations](../docs/source/_static/figures/xenium_correlations.png)


## 1. Configure your input

Install the notebook dependencies, then edit `DATA_DIR` and the input filenames.
Supply your own `.h5ad` files with unique spot and feature IDs. Target RNA fitting uses raw
counts; set `COUNTS_LAYER="counts"` when they are stored in a layer. Each fitted spot needs
positive total signal. The number of cell types (rows) in the reference determines `K`.

Running the fitting cells estimates new weights with the CPU fixed-reference solver.
`MAX_STEPS` limits optimization steps; the default is 100000 with tolerance `1e-10`.
Inspect the returned diagnostics to determine whether your fit converged. Large inputs
require enough memory for dense matrices.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display
from tutorials.common import correlation, align, spatial, pies, legend, save_figure, save_table
from spDecon.reproducibility import set_seed

DATA_DIR = Path("/path/to/your/data")
COUNTS_LAYER = None
BUILD_REFERENCE = True  # False to reuse an existing reference.h5ad.
REFERENCE_COUNTS_LAYER = "counts"  # None if scrna.X contains raw counts.
CELL_TYPE_COLUMN = "cell_type"
BALANCE_REFERENCE = True  # Study protocol; False for arithmetic raw-count means.
VERBOSE = True
MAX_STEPS = 100000
set_seed(1, deterministic=True)

TRUTH_CSV = None  # Optional: DATA_DIR / "fractions.csv"


## 2. Build a reference from annotated single cells

The path is **single-cell raw counts + cell-type labels → type profiles → COSG top50
marker union → target-gene alignment → `reference.h5ad`**. Set `BUILD_REFERENCE=False`
to skip construction and use an existing file. COSG 1.1.3 is included in
`requirements-notebooks.txt`; see the [COSG documentation](https://github.com/genecell/COSG).

Use matching gene identifiers in target and reference (for example, both gene symbols).
The example first intersects the complete shared gene panel. Raw counts are read from
`scrna.layers[REFERENCE_COUNTS_LAYER]`, or `scrna.X` when that setting is `None`.
Cell-type labels come from `scrna.obs[CELL_TYPE_COLUMN]`. Cells without any counts on
the shared panel are removed before calculating profiles or ranking markers.


In [ ]:
adata = ad.read_h5ad(DATA_DIR / "counts.h5ad")
if BUILD_REFERENCE:
    scrna = ad.read_h5ad(DATA_DIR / "scrna.h5ad")
    if not (scrna.obs_names.is_unique and scrna.var_names.is_unique and adata.var_names.is_unique):
        raise ValueError("Use unique cell IDs and gene IDs before preparing the reference")
    labels = scrna.obs[CELL_TYPE_COLUMN]
    if labels.isna().any() or labels.astype(str).str.strip().eq("").any():
        raise ValueError("Every reference cell needs a nonempty cell-type label")
    shared_genes = adata.var_names.intersection(scrna.var_names, sort=False)
    if not len(shared_genes):
        raise ValueError("The target and single-cell reference have no shared genes")
    raw_counts = scrna[:, shared_genes].to_df(layer=REFERENCE_COUNTS_LAYER)
    if not np.isfinite(raw_counts.to_numpy()).all() or (raw_counts.to_numpy() < 0).any():
        raise ValueError("Reference counts must be finite and nonnegative")
    keep = raw_counts.sum(axis=1) > 0
    print(f"Removing {(~keep).sum()} reference cells with zero shared-panel counts")
    raw_counts = raw_counts.loc[keep]
    cell_types = labels.loc[raw_counts.index].astype(str)
    if set(cell_types) != set(labels.astype(str)):
        raise ValueError("A reference type has no signal on the shared gene panel")
    display(cell_types.value_counts().rename("reference_cells"))


### 2.1 Calculate per-type reference profiles from raw counts

For arithmetic raw-count means, set `BALANCE_REFERENCE=False`. This computes
`raw_counts.groupby(cell_types, sort=False).mean()`: each row is one cell type.

The study used `BALANCE_REFERENCE=True`: divide each cell by its total on the **full
shared panel**, average these proportions within type, then multiply by that type's
mean raw-count total. In symbols, $A_{kg}=\overline{L}_k\,\mathrm{mean}_{c\in k}(X_{cg}/L_c)$.
This gives cells equal weight in profile shape while preserving the mean total per type.
Both options use raw counts and retain per-cell signal yields; compute them before
restricting to markers. Log-normalized values are used only for COSG ranking below.


In [ ]:
if BUILD_REFERENCE:
    from spDecon.reference import prepare_reference
    reference_full, _ = prepare_reference(raw_counts, cell_types, balanced=BALANCE_REFERENCE)
    display(reference_full.iloc[:5, :5])


### 2.2 Select the COSG top50 marker union

Following the study protocol, rank `log1p(10000 * counts / cell_total)` with COSG
`mu=1`. For each type, retain up to 50 positive, finite-score markers detected in at
least 10% of its cells and at least three cells. Take the deduplicated union without
padding. Marker selection uses reference cells and their labels; target composition
labels are not used. The union can contain fewer than `50 * K` genes.


In [ ]:
if BUILD_REFERENCE:
    import scanpy as sc
    import cosg

    ranked = ad.AnnData(raw_counts.copy())
    ranked.obs["cell_type"] = cell_types.astype("category")
    sc.pp.normalize_total(ranked, target_sum=1e4)
    sc.pp.log1p(ranked)
    cosg.cosg(ranked, groupby="cell_type", mu=1, use_raw=False,
        n_genes_user=min(50, ranked.n_vars), remove_lowly_expressed=True,
        expressed_pct=.1, expressed_min_num_cells_in_target_group=3,
        device="cpu", calculate_logfoldchanges=False, key_added="cosg")
    marker_names = pd.DataFrame(ranked.uns["cosg"]["names"])
    marker_scores = pd.DataFrame(ranked.uns["cosg"]["scores"])
    marker_table = marker_names.where(np.isfinite(marker_scores) & (marker_scores > 0))
    marker_union = pd.Index(pd.unique(marker_table.to_numpy().ravel())).dropna()
    if not len(marker_union):
        raise ValueError("No eligible COSG markers; check reference counts and cell-type labels")
    display(marker_table.head())
    print(f"COSG top50 union: {len(marker_union)} genes across {len(reference_full)} types")


### 2.3 Align to the target and save the reference

Keep marker genes in target order and save the unnormalized reference table.
The solver normalizes rows internally. Retaining the original row sums lets the
fitting cells convert molecular weights to cell fractions using fitted-panel yields.
The code writes `reference.h5ad` and `reference_markers.csv` under `DATA_DIR`.


In [ ]:
if BUILD_REFERENCE:
    panel = adata.var_names[adata.var_names.isin(marker_union)]
    reference = reference_full.loc[:, panel]
    if (reference.sum(axis=1) <= 0).any():
        raise ValueError("The marker panel leaves a reference type without signal")
    reference_adata = ad.AnnData(reference)
    reference_adata.obs["reference_signal_per_cell"] = reference.sum(axis=1)
    reference_adata.uns["reference_preparation"] = {
        "profiles": "cell_balanced" if BALANCE_REFERENCE else "arithmetic_raw_mean",
        "markers": "COSG top50 union", "mu": 1, "balanced": BALANCE_REFERENCE,
    }
    reference_adata.write_h5ad(DATA_DIR / "reference.h5ad")
    marker_table.to_csv(DATA_DIR / "reference_markers.csv", index=False)


## 3. Align the target and reference feature panels

Reference cell types determine `K`. Keep the unnormalized per-cell reference signals for the
cell-fraction conversion. Features are intersected and ordered by the target; empty intersections fail explicitly.


In [ ]:
reference_adata = ad.read_h5ad(DATA_DIR / "reference.h5ad")
genes = adata.var_names.intersection(reference_adata.var_names, sort=False)
if not len(genes):
    raise ValueError("The target and reference have no shared feature names")
adata = adata[:, genes].copy()
x = adata.to_df(layer=COUNTS_LAYER)
reference = reference_adata[:, genes].to_df()
xy = np.asarray(adata.obsm["spatial"])
sections = adata.obs["section_id"].to_numpy() if "section_id" in adata.obs else np.repeat("section_1", len(x))
K = len(reference)
display(reference.iloc[:5, :5])


## 4. Fit weights and convert to cell fractions


In [ ]:
from spDecon.spatial import spatial_graph
from spDecon.reference_spatial import fit as fit_fixed_reference
from spDecon.reference import cell_fractions
i, j, w, _ = spatial_graph(xy, sections)
p, b, history, diagnostics = fit_fixed_reference(
    x, reference, i, j, w, 0., tolerance=1e-10, max_steps=MAX_STEPS, verbose=VERBOSE)
fractions = pd.DataFrame(cell_fractions(p, reference.sum(axis=1).to_numpy()),
                         index=x.index, columns=reference.index)
display(diagnostics)
display(fractions.head())


## 5. Plot and export the estimated fractions


In [ ]:
fig, axes = plt.subplots(int(np.ceil(K/4)), 4, figsize=(12, 3*int(np.ceil(K/4))), squeeze=False)
for k, ax in enumerate(axes.ravel()):
    if k < K:
        art = spatial(ax, xy, fractions.iloc[:, k].to_numpy(), str(fractions.columns[k]), size=10)
    else:
        ax.axis("off")
fig.colorbar(art, ax=axes, label="Estimated cell fraction", shrink=.6)
save_figure(fig, "reference_based_fractions")
out = ROOT / "outputs/reference_based"
out.mkdir(parents=True, exist_ok=True)
fractions.to_csv(out / "cell_fractions.csv")


## 6. Optional evaluation

The optional fraction CSV must use the same cell-type names as the reference. If your annotations
use broader groups, aggregate the corresponding columns explicitly before evaluating them.


In [ ]:
if TRUTH_CSV is not None:
    truth = pd.read_csv(TRUTH_CSV, index_col=0)
    predicted = fractions.loc[truth.index, truth.columns]
    scores = pd.DataFrame({"type": truth.columns,
        "PCC": np.diag(correlation(predicted.to_numpy(), truth.to_numpy()))})
    display(save_table(scores, "reference_based_pcc"))
    print("RMSE:", np.sqrt(np.mean((predicted.to_numpy()-truth.to_numpy())**2)))
else:
    print("No fraction labels supplied; estimated fractions have been exported.")
